# E7 step 3 · The system's outputs from C4: damage maps, building table and "where to search first" hot spots for the 2,065 test photos (PC, no GPU)

Task **E7** (scoring our system's chain as one), step 3. Built 19 Nov 2026, after Abrar chose **C4** as the method that makes the damage map ([every option in plain words](../Task_Ledger.md#every-option-in-plain-words-written-19-nov-for-the-decision)).

**What C4 is.** A damage map marks every pixel of the after-photo as background, an intact building, a damaged building or a destroyed building. We have two methods that make one:
- **R11, our pixel model:** a U-Net that reads the before-photo and the after-photo together and marks every pixel. Touching buildings come out as one patch of pixels.
- **Find-then-grade:** SAM 3, Meta's image model, outlines each building on the before-photo. Then our grader, a small network that compares each building's before- and after-crop, labels the outline intact, damaged, destroyed or "not a building".

C4 uses R11 on the coarser xBD photos (one pixel is 1.25–2.8 m of ground) and find-then-grade on the sharper photos (about 0.3 m per pixel, such as Turkey's). Here the kind of photo comes from its file name, by E1's rule, which agrees with E6's labels on all 2,067 test pairs.

**Why this step.** The written reports (T2) need one building table and one set of hot spots, both from the chosen method. Until now they came from different models on different photos: H2's table from R11 and T1 v3's hot spots from R9, both on validation photos, and the two shared only 7 photos.

**What it makes,** only from files E7 step 2 saved. Nothing is retrained, and nothing runs on a GPU.
1. **C4's damage map for each of the 2,065 test photos.** On xBD photos it's R11's saved map. On the others it's find-then-grade's saved map with grader seed 42, the seed fixed for the table on 4 Nov, before any test score.
2. **The building table** (`buildings_c4.sqlite`). It keeps H2's columns, so the report bridge reads it unchanged, and adds two things:
   - **On the sharper photos, one row per building SAM 3 outlined** (column `kind` = `outline`), so touching buildings stay separate. E7 step 2's table merged them, because it built rows by joining touching pixels of one class, as H2 does.
   - **On xBD photos, one row per connected patch of R11's pixels** (`kind` = `patch`), exactly as H2 builds rows. A patch can hold several touching buildings, so a report must never call a patch "one building".
   - The `images` table also records each photo's source and method.
3. **The hot spots and safe areas** (`hotspots.json`), using T1 v3's code and settings unchanged, on C4's maps.
   - **Danger:** destroyed pixels (weight 1.0) and damaged pixels (0.5), blurred.
   - **Priority:** danger weighted by closeness to damage.
   - **Hot spots:** the 3 strongest zones of high priority, each with a marked point.
   - **Safe areas:** the 3 open points farthest from any damage.
4. **Beside it, deciding nothing:** C4's counting on the benchmark's building-counting questions, from the counts E7 step 2 saved.

**Checks (the notebook stops if one fails)**
1. The file-name rule gives E6's source for every test pair.
2. Seed 42's map, rebuilt from SAM 3's saved outlines and the graders' saved scores, equals the saved map on at least 99.99% of pixels.
3. On E6's 2,007 clean pairs the scores reproduce the published ones within 0.0005:
   - R11: 0.3984 destroyed IoU and 0.5721 mIoU (E7 step 2);
   - the three graders' find-then-grade maps (E7 step 2);
   - C4 as the mean of the three seeds: 0.4296 destroyed IoU, 0.5555 mIoU, 0.6221 building IoU, 0.3231 on Turkey and 0.4610 without Turkey (E7 step 2c).
4. H2's row code, run on the saved seed-42 maps, rebuilds E7 step 2's saved table exactly.
5. T1 v3's self-checks and the table's self-checks pass. In particular, two touching outlines stay two rows.
6. Ridita's report bridge, `disasteller_inputs.py` (unchanged), reads the new table and hot-spot file for one xBD photo and one other photo.

**Expectation (fixed before the run)**
- All six checks pass.
- On the sharper photos, the table has more buildings than E7 step 2's table on the same photos, because touching buildings no longer merge: about 5–20% more (a guess).
- On the clean pairs, the hot spots' marked points land within 20 px of true damage about as often as T1 v3's did on validation with R9's maps (0.736), or more often, because C4 finds more destroyed buildings on Turkey: 0.70–0.85.
- Counting, beside: within about 1.5 points of R11 (32.53%) and find-then-grade (32.7%) on E5's clean questions. E7 step 2 found the two level.

**How to run:** on the PC, from `Essential_task_delegation/Champion_Task/`, in Jupyter with the `cse499` kernel, which has OpenCV since 19 Nov. With `SMOKE_TEST = True` it runs 40 photos (20 xBD, 20 other) and skips check 3, which needs all 2,007 clean pairs. The full run takes about 30–40 min, estimated.

**Outputs** in `E7-results/step3/`:
- `README.md`, `config.json`;
- `buildings_c4.sqlite`, the table, and `table_summary.csv`;
- `hotspots.json`, `hotspots_per_image.csv`, `hotspots_summary.csv`;
- `scores.csv`: the reproduced scores, and C4's per seed;
- `counting.csv`;
- `overlays/`: 20 photos to check by eye;
- `maps/` and `c4_maps.zip`, both git-ignored.

In [1]:
# ── Cell 1: Configuration and inputs (PC, no GPU, nothing downloaded) ──
import io
import json
import os
import re
import sqlite3
import sys
import time
import zipfile
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.stats import binomtest
from tqdm.auto import tqdm

SMOKE_TEST = False             # True = 40 photos (20 xBD, 20 other, seed 42); check 3 needs all 2,007 clean pairs
CT = Path(os.environ.get("CHAMPION_TASK", Path.cwd()))     # run from Champion_Task
OUT_DIR = Path(os.environ.get("E7S3_OUT", CT / "E7-results" / "step3"))
S2, S2C, E6 = CT / "E7-results" / "step2", CT / "E7-results" / "step2c", CT / "E6-results" / "E6"
BRIDGE = CT.parent / "Offline_RAG_KR2" / "disasteller_offline"
INPUTS = [S2 / f for f in ("r11_tta_maps.zip", "chain_s42_maps.zip", "sam3_pre_grid2.zip", "instances_test.csv",
                           "buildings_chain_s42.sqlite", "map_scores.csv", "per_question.csv", "counting_scores.csv")]
INPUTS += [E6 / "test_pairs.csv", E6 / "test_masks_4class.zip", S2C / "test_confirmation.csv", BRIDGE / "disasteller_inputs.py"]
missing = [str(p) for p in INPUTS if not p.exists()]
assert not missing, f"Run this notebook from Champion_Task; missing: {missing}"

NUM_CLASSES = 4
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
CLASS_NAME = {1: "intact", 2: "damaged", 3: "destroyed"}
LABELS = ["intact", "damaged", "destroyed", "none"]          # the graders' four labels, in their order
SEEDS, TABLE_SEED = (42, 43, 44), 42                         # E7 step 2: the table's seed, fixed 4 Nov before any test score
BIAS_FIXED = {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}   # E7 step 2
MIN_AREA = 20                                                # R16's smallest building piece, also H2's table rule
TOL, MIN_REBUILD_AGREEMENT = 0.0005, 0.9999
C4_NAME = "C4, R11 on xBD images, find-then-grade on the others"    # its row in E7 step 2c's test_confirmation.csv
# T1 v3's settings, unchanged
SIGMA_PX, DAMAGED_WEIGHT, SAFE_SCALE_PX, HOT_THRESHOLD, MIN_SPOT_PX = 25, 0.5, 200, 0.05, 500
POINT_EDGE_MARGIN_PX, SAFE_MIN_GAP_PX, SAFE_EDGE_MARGIN_PX, POINT_RADIUS_PX = 32, 150, 100, 20
DISPLAY_MAX, N_CHECK = 0.5, 10
T1V3_POINT_ON_DAMAGE = 0.736                                 # T1 v3 on the 645 validation photos, from R9's maps
XBD_NAME, OTHER_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster"), re.compile(r"^(.+?)_post")


def event_and_source(stem):
    """E1's rule: 'hurricane_florence_00000261_post_disaster' -> ('hurricane_florence', 'xBD'),
    'turkey_earthquake5_post_12' -> ('turkey_earthquake', 'other')."""
    m = XBD_NAME.match(stem)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(stem)
    return re.sub(r"\d+$", "", m.group(1) if m else stem), "other"


pairs = pd.read_csv(E6 / "test_pairs.csv")
rule = pairs["after_image"].map(lambda n: event_and_source(Path(n).stem)[1])
assert (rule == pairs["source"]).all(), "check 1 failed: the file-name rule disagrees with E6's source"
Z = {k: zipfile.ZipFile(S2 / f) for k, f in (("r11", "r11_tta_maps.zip"), ("chain", "chain_s42_maps.zip"),
                                              ("sam", "sam3_pre_grid2.zip"))}
MASKS = zipfile.ZipFile(E6 / "test_masks_4class.zip")
ALL_NAMES = sorted(set(Z["r11"].namelist()) & set(Z["chain"].namelist()) & set(Z["sam"].namelist()))
clean = set(pairs.loc[pairs["clean"] == True, "after_image"])
mask_file = dict(zip(pairs["after_image"], pairs["mask_file"]))
assert len(ALL_NAMES) == 2065 and len(clean) == 2007 and clean <= set(ALL_NAMES), (len(ALL_NAMES), len(clean))
source_of = {n: event_and_source(Path(n).stem)[1] for n in ALL_NAMES}
names = ALL_NAMES
if SMOKE_TEST:
    rng = np.random.default_rng(42)
    names = sorted(rng.choice([n for n in ALL_NAMES if source_of[n] == "xBD"], 20, replace=False).tolist()
                   + rng.choice([n for n in ALL_NAMES if source_of[n] == "other"], 20, replace=False).tolist())
logp_cols = {s: [f"logp_{label}_s{s}" for label in LABELS] for s in SEEDS}
inst = pd.read_csv(S2 / "instances_test.csv", usecols=["image", "id", *[c for s in SEEDS for c in logp_cols[s]]])
inst_of = dict(tuple(inst.groupby("image")))
published = pd.read_csv(S2 / "map_scores.csv")
c4_published = pd.read_csv(S2C / "test_confirmation.csv").set_index("method").loc[C4_NAME]
(OUT_DIR / "maps").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "overlays").mkdir(exist_ok=True)
t_start = time.time()
print(f"✓ E7 step 3{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {len(names)} test photos "
      f"({sum(source_of[n] == 'xBD' for n in names)} xBD, {sum(source_of[n] == 'other' for n in names)} other), "
      f"{len(clean & set(names))} of them clean; {len(inst):,} SAM 3 outlines with three graders' scores")
print(f"✓ Check 1: the file-name rule gives E6's source for all {len(pairs):,} test pairs")
print(f"  Outputs: {OUT_DIR}")

✓ E7 step 3: 2065 test photos (1326 xBD, 739 other), 2007 of them clean; 138,725 SAM 3 outlines with three graders' scores
✓ Check 1: the file-name rule gives E6's source for all 2,067 test pairs
  Outputs: D:\CSE499AB_project\Essential_task_delegation\Champion_Task\E7-results\step3


In [2]:
# ── Cell 2: The maps, the table's rows (E7 step 2's and H2's code), T1 v3's hot spots, and self-checks ──
def read_png(z, name, flags=cv2.IMREAD_UNCHANGED):
    return cv2.imdecode(np.frombuffer(z.read(name), np.uint8), flags)


def to_class(labels):
    """Grader labels -> map classes: intact, damaged, destroyed -> 1-3; "none" -> 0 (background). E7 step 2's."""
    return np.where(labels < 3, labels + 1, 0)


def graded_maps(name, ids):
    """Each seed's find-then-grade map, rebuilt as E7 step 2 painted it: every SAM 3 outline gets the class of its
    top label after the seed's fixed biases."""
    g = inst_of.get(name)
    maps = {}
    for seed in SEEDS:
        lut = np.zeros(int(ids.max()) + 1, np.uint8)
        if g is not None:
            lut[g["id"].to_numpy()] = to_class((g[logp_cols[seed]].to_numpy() + np.array(BIAS_FIXED[seed])).argmax(1))
        maps[seed] = lut[ids]
    return maps


def class_of_outline(ids, painted):
    """{outline id: its class in a painted map} (each outline is painted in one class)."""
    size = int(ids.max()) + 1
    counts = np.bincount(ids.ravel().astype(np.int64) * NUM_CLASSES + painted.ravel(),
                         minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    present = np.nonzero(counts.sum(1))[0]
    return {int(i): int(counts[i].argmax()) for i in present if i > 0}


def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction. E7 step 2's."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged). E7 step 2's."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision, recall = tp / (tp + fp + eps), tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": round(float(iou.mean()), 4), **{f"{c.lower()}_iou": round(float(iou[i]), 4) for i, c in enumerate(CLASS_NAMES)},
            "building_iou": round(float(building_iou), 4), "destroyed_precision": round(float(precision[3]), 4),
            "destroyed_recall": round(float(recall[3]), 4)}


# H2's table, as E7 step 2 built it (same columns, same rows), plus each row's kind
COLUMNS = [("image", "TEXT"), ("building_id", "INTEGER"), ("class_id", "INTEGER"), ("class_name", "TEXT"), ("area_px", "INTEGER"),
           ("centroid_x", "REAL"), ("centroid_y", "REAL"), ("bbox_x", "INTEGER"), ("bbox_y", "INTEGER"), ("bbox_w", "INTEGER"),
           ("bbox_h", "INTEGER"), ("dist_to_nearest_destroyed_px", "REAL")]


def create_schema(conn):
    cols = ",\n  ".join(f"{n} {t}" for n, t in COLUMNS + [("kind", "TEXT")])
    conn.execute(f"CREATE TABLE buildings (\n  {cols},\n  PRIMARY KEY (image, building_id))")
    conn.execute("CREATE TABLE images (image TEXT PRIMARY KEY, width INTEGER, height INTEGER, source TEXT, method TEXT)")
    conn.execute("CREATE INDEX idx_buildings_class ON buildings (image, class_name)")


def add_nearest_destroyed(blobs):
    """E7 step 2's (H2's): each row's distance, centroid to centroid, to the nearest other destroyed row."""
    for b in blobs:
        b["dist_to_nearest_destroyed_px"] = None
    dest = [b for b in blobs if b["class_id"] == 3]
    if not dest:
        return
    tree = cKDTree(np.array([[b["centroid_x"], b["centroid_y"]] for b in dest]))
    for b in blobs:
        p = (b["centroid_x"], b["centroid_y"])
        if b["class_id"] == 3:
            if len(dest) < 2:
                continue
            d = float(tree.query(p, k=2)[0][1])      # the nearest hit is the blob itself, at distance 0
        else:
            d = float(tree.query(p, k=1)[0])
        b["dist_to_nearest_destroyed_px"] = round(d, 1)


def blobs_of(pred):
    """E7 step 2's (H2's) rows for one 0-3 map: 8-connected blobs per class, minus blobs under MIN_AREA."""
    out = []
    for cls, cname in CLASS_NAME.items():
        n, _, stats, cents = cv2.connectedComponentsWithStats((pred == cls).astype(np.uint8), connectivity=8)
        for k in range(1, n):
            area = int(stats[k, cv2.CC_STAT_AREA])
            if area < MIN_AREA:
                continue
            out.append({"class_id": cls, "class_name": cname, "area_px": area,
                        "centroid_x": round(float(cents[k][0]), 1), "centroid_y": round(float(cents[k][1]), 1),
                        "bbox_x": int(stats[k, cv2.CC_STAT_LEFT]), "bbox_y": int(stats[k, cv2.CC_STAT_TOP]),
                        "bbox_w": int(stats[k, cv2.CC_STAT_WIDTH]), "bbox_h": int(stats[k, cv2.CC_STAT_HEIGHT])})
    for i, b in enumerate(out, 1):
        b["building_id"] = i
    add_nearest_destroyed(out)
    return out


def outline_rows(ids, cls_of):
    """C4's rows on the sharper photos: one per SAM 3 outline graded as a building, at least MIN_AREA px, with H2's
    columns computed on the outline's own pixels (the same formulas as OpenCV's for a blob)."""
    size = int(ids.max()) + 1
    flat = ids.ravel()
    area = np.bincount(flat, minlength=size)
    yy, xx = np.indices(ids.shape)
    sx = np.bincount(flat, weights=xx.ravel(), minlength=size)
    sy = np.bincount(flat, weights=yy.ravel(), minlength=size)
    boxes = ndimage.find_objects(ids)
    out = []
    for i in sorted(cls_of):
        cls = cls_of[i]
        if cls == 0 or area[i] < MIN_AREA:
            continue
        rows_sl, cols_sl = boxes[i - 1]
        out.append({"class_id": cls, "class_name": CLASS_NAME[cls], "area_px": int(area[i]),
                    "centroid_x": round(float(sx[i] / area[i]), 1), "centroid_y": round(float(sy[i] / area[i]), 1),
                    "bbox_x": cols_sl.start, "bbox_y": rows_sl.start,
                    "bbox_w": cols_sl.stop - cols_sl.start, "bbox_h": rows_sl.stop - rows_sl.start})
    for k, b in enumerate(out, 1):
        b["building_id"] = k
    add_nearest_destroyed(out)
    return out


def insert_image(conn, name, height, width, source, method, kind, rows):
    conn.execute("INSERT INTO images VALUES (?, ?, ?, ?, ?)", (name, width, height, source, method))
    cols = [n for n, _ in COLUMNS] + ["kind"]
    conn.executemany(f"INSERT INTO buildings ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})",
                     [tuple(name if c == "image" else kind if c == "kind" else b[c] for c in cols) for b in rows])


# T1 v3's map functions, unchanged
EIGHT = np.ones((3, 3), bool)


def survivability_maps(mask):
    """Returns danger, safety and priority (float32, 0-1), and the distance in px to the nearest damage."""
    weight = (mask == 3).astype(np.float32) + DAMAGED_WEIGHT * (mask == 2)
    danger = ndimage.gaussian_filter(weight, SIGMA_PX, mode="mirror", truncate=4.0)   # = v1's cv2.GaussianBlur
    damage = mask >= 2
    dist = (ndimage.distance_transform_edt(~damage).astype(np.float32) if damage.any()
            else np.full(mask.shape, np.inf, np.float32))
    safety = np.minimum(dist / SAFE_SCALE_PX, 1).astype(np.float32)
    return danger, safety, danger * (1 - safety), dist


def compass(x, y, w, h):
    """Where a point sits in the image (top = north): 'north-east', 'west', 'centre', ..."""
    row = ("north", "", "south")[min(int(3 * y / h), 2)]
    col = ("west", "", "east")[min(int(3 * x / w), 2)]
    return "-".join(p for p in (row, col) if p) or "centre"


def hot_spots(priority, k=3):
    """The k strongest zones (connected, priority >= HOT_THRESHOLD, at least MIN_SPOT_PX pixels), ranked by
    total priority. Each is marked at its strongest point at least POINT_EDGE_MARGIN_PX from the image edge
    (anywhere in the zone if none of it is that far in), the one nearest the zone's middle if several tie.
    Also returns the label image ('label' is each zone's id in it)."""
    labels, n = ndimage.label(priority >= HOT_THRESHOLD, structure=EIGHT)
    if n == 0:
        return [], labels
    area = np.bincount(labels.ravel(), minlength=n + 1)
    mass = np.bincount(labels.ravel(), weights=priority.ravel(), minlength=n + 1)
    mass[0] = -1                                    # label 0 is everything below the threshold
    mass[area < MIN_SPOT_PX] = -1                   # specks
    h, w = priority.shape
    m = POINT_EDGE_MARGIN_PX
    spots = []
    for i in np.argsort(-mass)[:k]:
        if mass[i] < 0:
            break
        ys, xs = np.nonzero(labels == i)
        peak = float(priority[ys, xs].max())
        inner = (ys >= m) & (ys < h - m) & (xs >= m) & (xs < w - m)
        if inner.any():                             # v3: keep the mark off the image edge
            ys, xs = ys[inner], xs[inner]
        values = priority[ys, xs]
        top = values >= values.max() - 1e-6
        j = np.argmin((ys[top] - ys.mean()) ** 2 + (xs[top] - xs.mean()) ** 2)
        x, y = int(xs[top][j]), int(ys[top][j])
        spots.append({"rank": len(spots) + 1, "label": int(i), "point_xy": [x, y], "area_px": int(area[i]),
                      "peak": round(peak, 3), "edge_px": int(min(x, y, w - 1 - x, h - 1 - y)),
                      "where": compass(x, y, w, h)})
    return spots, labels


def safe_areas(dist, mask, k=3):
    """The k open (background) points farthest from any damage, at least SAFE_MIN_GAP_PX apart and at least
    SAFE_EDGE_MARGIN_PX from the image edge, because damage just outside the picture is invisible."""
    if np.isinf(dist).all():
        return []                                   # no damage predicted: the whole image is safe
    h, w = mask.shape
    m = SAFE_EDGE_MARGIN_PX
    inner = (slice(m, h - m), slice(m, w - m))
    score = np.full(mask.shape, -1.0)
    score[inner] = np.where(mask[inner] == 0, dist[inner], -1.0)
    yy, xx = np.ogrid[:h, :w]
    areas = []
    for rank in range(1, k + 1):
        y, x = np.unravel_index(np.argmax(score), score.shape)
        if score[y, x] < 0:
            break
        areas.append({"rank": rank, "xy": [int(x), int(y)], "distance_px": int(score[y, x]),
                      "edge_px": int(min(x, y, w - 1 - x, h - 1 - y)), "where": compass(x, y, w, h)})
        score[(yy - y) ** 2 + (xx - x) ** 2 < SAFE_MIN_GAP_PX ** 2] = -1
    return areas


def save_map(m, path):
    """256 x 256 copy (area average), value x 255."""
    small = np.asarray(Image.fromarray(np.clip(m, 0, 1).astype(np.float32)).resize((256, 256), Image.BOX))
    Image.fromarray(np.round(small * 255).astype(np.uint8)).save(path)


def near(gt, xy, classes):
    """Is one of these ground-truth classes within POINT_RADIUS_PX of the point?"""
    x, y, r = xy[0], xy[1], POINT_RADIUS_PX
    return bool(np.isin(gt[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1], classes).any())


# T1 v3's self-checks: one destroyed block -> one hot spot, marked on the block; safe areas far from it and off the
# edge; a speck is not a hot spot; a zone strongest at the image edge is marked 32 px in; no damage -> no safe areas
_m = np.zeros((600, 600), np.uint8)
_m[280:320, 280:320] = 3
_d, _s, _p, _dist = survivability_maps(_m)
_spots, _ = hot_spots(_p)
assert len(_spots) == 1 and _spots[0]["where"] == "centre"
assert _m[_spots[0]["point_xy"][1], _spots[0]["point_xy"][0]] == 3
_safe = safe_areas(_dist, _m)
assert len(_safe) == 3 and all(a["distance_px"] > 150 and a["edge_px"] >= SAFE_EDGE_MARGIN_PX for a in _safe)
_p2 = np.zeros((300, 300), np.float32)
_p2[10:20, 10:20] = 0.06                        # a 100-pixel speck
_p2[100:150, 100:150] = 0.5                     # a real zone
assert [s["area_px"] for s in hot_spots(_p2)[0]] == [2500]
_p3 = np.zeros((300, 300), np.float32)
_p3[0:60, 100:160] = np.linspace(1.0, 0.5, 60, dtype=np.float32)[:, None]   # strongest on the top edge
_s3 = hot_spots(_p3)[0][0]
assert _s3["point_xy"][1] == POINT_EDGE_MARGIN_PX and _s3["edge_px"] >= POINT_EDGE_MARGIN_PX
assert safe_areas(np.full((5, 5), np.inf, np.float32), np.zeros((5, 5), np.uint8)) == []

# The table's self-checks: two touching outlines stay two rows, while the same pixels as one map make one patch;
# an outline under MIN_AREA or graded "not a building" gives no row; distances run centroid to centroid
_ids = np.zeros((60, 60), np.int32)
_ids[10:30, 10:20], _ids[10:30, 20:30], _ids[40:42, 40:42] = 1, 2, 3
_rows = outline_rows(_ids, {1: 1, 2: 1, 3: 1})
assert [r["area_px"] for r in _rows] == [200, 200] and _rows[0]["bbox_w"] == 10 and _rows[0]["centroid_x"] == 14.5
assert len(blobs_of((_ids > 0).astype(np.uint8))) == 1
assert [r["class_name"] for r in outline_rows(_ids, {1: 3, 2: 0, 3: 1})] == ["destroyed"]
assert outline_rows(_ids, {1: 3, 2: 1, 3: 1})[1]["dist_to_nearest_destroyed_px"] == 10.0
assert class_of_outline(_ids, np.where(_ids == 1, 3, np.where(_ids > 1, 1, 0)).astype(np.uint8)) == {1: 3, 2: 1, 3: 1}
print("✓ Functions ready; T1 v3's and the table's self-checks passed")

✓ Functions ready; T1 v3's and the table's self-checks passed


In [3]:
# ── Cell 3: Check 4: H2's row code, run on the saved seed-42 maps, rebuilds E7 step 2's saved table exactly ──
saved_table = sqlite3.connect(S2 / "buildings_chain_s42.sqlite")
cols = [n for n, _ in COLUMNS]
bad = []
for name in tqdm(names, desc="Check 4"):
    mine = [tuple(name if c == "image" else b[c] for c in cols)
            for b in blobs_of(read_png(Z["chain"], name, cv2.IMREAD_GRAYSCALE))]
    theirs = saved_table.execute(f"SELECT {', '.join(cols)} FROM buildings WHERE image = ? ORDER BY building_id",
                                 (name,)).fetchall()
    if mine != theirs:
        bad.append(name)
assert not bad, f"check 4 failed on {len(bad)} photos, e.g. {bad[:3]}"
print(f"✓ Check 4: H2's row code rebuilds E7 step 2's table on all {len(names)} photos, row for row")

Check 4:   0%|          | 0/2065 [00:00<?, ?it/s]

✓ Check 4: H2's row code rebuilds E7 step 2's table on all 2065 photos, row for row


In [4]:
# ── Cell 4: C4's map for every photo, the confusion counts on the clean pairs, and C4's building table ──
DB = OUT_DIR / "buildings_c4.sqlite"
DB.unlink(missing_ok=True)
conn = sqlite3.connect(DB)
create_schema(conn)
METHODS = ["R11", *[f"chain_s{s}" for s in SEEDS], *[f"C4_s{s}" for s in SEEDS]]
cms = {(scope, m): np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)
       for scope in ("clean pairs", "Turkey only", "clean pairs, without Turkey") for m in METHODS}
rebuilt_same, rebuilt_px = 0, 0
c4_zip = zipfile.ZipFile(OUT_DIR / "c4_maps.zip", "w", zipfile.ZIP_STORED)
per_photo = []
for name in tqdm(names, desc="C4 maps and table"):
    r11 = read_png(Z["r11"], name, cv2.IMREAD_GRAYSCALE)
    saved42 = read_png(Z["chain"], name, cv2.IMREAD_GRAYSCALE)
    h, w = r11.shape
    ids = read_png(Z["sam"], name)
    if ids.shape != (h, w):
        ids = cv2.resize(ids, (w, h), interpolation=cv2.INTER_NEAREST)
    ids = ids.astype(np.int64)
    chain = graded_maps(name, ids)
    rebuilt_same += int((chain[TABLE_SEED] == saved42).sum())
    rebuilt_px += saved42.size
    event, source = event_and_source(Path(name).stem)
    if name in clean:
        gt = read_png(MASKS, mask_file[name], cv2.IMREAD_GRAYSCALE)
        if gt.shape != (h, w):
            gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)
        scopes = ["clean pairs", "Turkey only" if event == "turkey_earthquake" else "clean pairs, without Turkey"]
        for m, p in [("R11", r11), *[(f"chain_s{s}", chain[s]) for s in SEEDS],
                     *[(f"C4_s{s}", r11 if source == "xBD" else chain[s]) for s in SEEDS]]:
            cm = confusion(gt, p)
            for scope in scopes:
                cms[(scope, m)] += cm
    c4 = r11 if source == "xBD" else saved42           # the output map: R11's, or find-then-grade's as saved (seed 42)
    c4_zip.writestr(name, cv2.imencode(".png", c4)[1].tobytes())
    if source == "xBD":
        rows, kind, method = blobs_of(r11), "patch", "R11 (pixel model)"
    else:
        rows, kind, method = outline_rows(ids, class_of_outline(ids, saved42)), "outline", "find-then-grade, grader seed 42"
    insert_image(conn, name, h, w, source, method, kind, rows)
    e7_rows = saved_table.execute("SELECT class_name, COUNT(*) FROM buildings WHERE image = ? GROUP BY class_name",
                                  (name,)).fetchall()
    per_photo.append({"image": name, "event": event, "source": source, "kind": kind, "clean": name in clean,
                      **{c: sum(r["class_name"] == c for r in rows) for c in CLASS_NAME.values()},
                      **{f"e7_step2_{c}": dict(e7_rows).get(c, 0) for c in CLASS_NAME.values()}})
conn.commit()
c4_zip.close()
TABLE = pd.DataFrame(per_photo)
agreement = rebuilt_same / rebuilt_px
assert agreement >= MIN_REBUILD_AGREEMENT, f"check 2 failed: seed 42's rebuilt map matches the saved one on {agreement:.6f} of pixels"
n_rows = conn.execute("SELECT COUNT(*) FROM buildings").fetchone()[0]
by_kind = dict(conn.execute("SELECT kind, COUNT(*) FROM buildings GROUP BY kind").fetchall())
print(f"✓ Check 2: seed 42's rebuilt map matches the saved one on {agreement:.6%} of pixels")
print(f"✓ C4's table: {n_rows:,} rows {by_kind} on {len(names)} photos; C4's maps in {OUT_DIR / 'c4_maps.zip'}")

C4 maps and table:   0%|          | 0/2065 [00:00<?, ?it/s]

✓ Check 2: seed 42's rebuilt map matches the saved one on 99.999566% of pixels
✓ C4's table: 125,439 rows {'outline': 57849, 'patch': 67590} on 2065 photos; C4's maps in D:\CSE499AB_project\Essential_task_delegation\Champion_Task\E7-results\step3\c4_maps.zip


In [5]:
# ── Cell 5: Check 3: the scores reproduce the published ones; C4's scores per seed ──
SCORES = pd.DataFrame([{"scope": scope, "method": m, **metrics(cm)} for (scope, m), cm in cms.items()])


def score(scope, method, key):
    return SCORES[(SCORES["scope"] == scope) & (SCORES["method"] == method)][key].item()


def mean_of_three(scope, key):
    return round(float(np.mean([score(scope, f"C4_s{s}", key) for s in SEEDS])), 4)


pub = published[published["scope"] == "clean pairs"].set_index("method")
rows = [{"what": f"{m}: {key}", "rebuilt": score("clean pairs", m, key), "published": float(pub.loc[pm, key]),
         "from": "E7 step 2, map_scores.csv"}
        for m, pm in [("R11", "R11_tta"), *[(f"chain_s{s}", f"chain_s{s}") for s in SEEDS]]
        for key in ("destroyed_iou", "miou", "building_iou")]
rows += [{"what": f"C4, mean of three seeds: {label}", "rebuilt": mean_of_three(scope, key),
          "published": float(c4_published[label]), "from": "E7 step 2c, test_confirmation.csv"}
         for label, scope, key in [("destroyed_iou (all)", "clean pairs", "destroyed_iou"), ("miou", "clean pairs", "miou"),
                                   ("building_iou", "clean pairs", "building_iou"),
                                   ("destroyed_iou (Turkey)", "Turkey only", "destroyed_iou"),
                                   ("destroyed_iou (without Turkey)", "clean pairs, without Turkey", "destroyed_iou")]]
CHECK3 = pd.DataFrame(rows)
CHECK3["difference"] = (CHECK3["rebuilt"] - CHECK3["published"]).round(4)
CHECK3["pass"] = CHECK3["difference"].abs() <= TOL
print(CHECK3.to_string(index=False))
if SMOKE_TEST:
    print("\nSmoke test: check 3 needs all 2,007 clean pairs, so it isn't applied")
else:
    assert CHECK3["pass"].all(), "check 3 failed: a rebuilt score differs from the published one by more than 0.0005"
    print("\n✓ Check 3: every rebuilt score is within 0.0005 of the published one")
print("\nC4 on the clean pairs, per grader seed:")
print(SCORES[SCORES["method"].str.startswith("C4_")].to_string(index=False))

                                                   what  rebuilt  published                              from  difference  pass
                                     R11: destroyed_iou   0.3984     0.3984         E7 step 2, map_scores.csv      0.0000  True
                                              R11: miou   0.5721     0.5721         E7 step 2, map_scores.csv      0.0000  True
                                      R11: building_iou   0.6944     0.6944         E7 step 2, map_scores.csv      0.0000  True
                               chain_s42: destroyed_iou   0.4055     0.4055         E7 step 2, map_scores.csv      0.0000  True
                                        chain_s42: miou   0.5180     0.5179         E7 step 2, map_scores.csv      0.0001  True
                                chain_s42: building_iou   0.5395     0.5395         E7 step 2, map_scores.csv      0.0000  True
                               chain_s43: destroyed_iou   0.3982     0.3983         E7 step 2, map_score

In [6]:
# ── Cell 6: T1 v3's hot spots and safe areas on C4's maps, and the ground-truth checks on the clean pairs ──
C4Z = zipfile.ZipFile(OUT_DIR / "c4_maps.zip")
records, rows = [], []
for name in tqdm(names, desc="Hot spots"):
    pred = read_png(C4Z, name, cv2.IMREAD_GRAYSCALE)
    stem = Path(name).stem
    event, source = event_and_source(stem)
    danger, safety, priority, dist = survivability_maps(pred)
    spots, labels = hot_spots(priority)
    gt = None
    if name in clean:
        gt = read_png(MASKS, mask_file[name], cv2.IMREAD_GRAYSCALE)
        if gt.shape != pred.shape:
            gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST)
    for s in spots:
        zone = labels == s.pop("label")
        if gt is None:                                 # not a clean pair: no trusted truth to check against
            s.update(on_gt_destroyed=None, on_gt_damage=None, point_on_gt_destroyed=None, point_on_gt_damage=None)
            continue
        s["on_gt_destroyed"] = bool((zone & (gt == 3)).any())
        s["on_gt_damage"] = bool((zone & (gt >= 2)).any())
        s["point_on_gt_destroyed"] = near(gt, s["point_xy"], [3])
        s["point_on_gt_damage"] = near(gt, s["point_xy"], [2, 3])
    safe = safe_areas(dist, pred)
    for a in safe:
        a["on_gt_building"] = None if gt is None else bool(gt[a["xy"][1], a["xy"][0]] > 0)
    records.append({"image": name, "event": event, "source": source,
                    "method": "R11" if source == "xBD" else "find-then-grade", "clean": name in clean,
                    "pred_damaged_px": int((pred == 2).sum()), "pred_destroyed_px": int((pred == 3).sum()),
                    "hot_spots": spots, "safe_areas": safe})
    if gt is not None:
        gt_spots, gt_labels = hot_spots(survivability_maps(gt)[2])
        rows.append({"image": name, "event": event, "source": source, "gt_destroyed_px": int((gt == 3).sum()),
                     "hot_spots": len(spots), "on_gt_destroyed": sum(s["on_gt_destroyed"] for s in spots),
                     "on_gt_damage": sum(s["on_gt_damage"] for s in spots),
                     "point_on_gt_destroyed": sum(s["point_on_gt_destroyed"] for s in spots),
                     "point_on_gt_damage": sum(s["point_on_gt_damage"] for s in spots),
                     "points_on_edge": sum(s["edge_px"] <= 5 for s in spots),
                     "safe_areas": len(safe), "safe_on_gt_building": sum(a["on_gt_building"] for a in safe),
                     "gt_hot_spots": len(gt_spots),
                     "gt_on_destroyed": sum(bool(((gt_labels == s["label"]) & (gt == 3)).any()) for s in gt_spots)})
    for kind, m in (("kde", danger), ("safe", safety), ("priority", priority)):
        save_map(m, OUT_DIR / "maps" / f"{stem}_{kind}.png")
per_image = pd.DataFrame(rows)


def hit_rates(df):
    """T1 v3's."""
    n = max(int(df["hot_spots"].sum()), 1)
    return {"images": len(df), "hot_spots": int(df["hot_spots"].sum()),
            "on_gt_destroyed": round(df["on_gt_destroyed"].sum() / n, 3),
            "ceiling_from_gt_mask": round(df["gt_on_destroyed"].sum() / max(int(df["gt_hot_spots"].sum()), 1), 3),
            "on_gt_damaged_or_destroyed": round(df["on_gt_damage"].sum() / n, 3),
            "point_on_gt_destroyed": round(df["point_on_gt_destroyed"].sum() / n, 3),
            "point_on_gt_damage": round(df["point_on_gt_damage"].sum() / n, 3),
            "points_within_5px_of_edge": round(df["points_on_edge"].sum() / n, 3),
            "safe_on_gt_building": round(df["safe_on_gt_building"].sum() / max(int(df["safe_areas"].sum()), 1), 3)}


HOT_SUMMARY = pd.DataFrame([
    {"images_from": population, "source": source, **hit_rates(g)}
    for population, df in [("clean pairs", per_image), ("with destroyed", per_image[per_image["gt_destroyed_px"] > 0])]
    for source, g in [("all", df), *df.groupby("source")]])
with open(OUT_DIR / "hotspots.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=1)
HOT_SUMMARY.to_csv(OUT_DIR / "hotspots_summary.csv", index=False)
per_image.to_csv(OUT_DIR / "hotspots_per_image.csv", index=False)
print(HOT_SUMMARY.to_string(index=False))
HOT_ALL = hit_rates(per_image)
print(f"\n✓ Hot spots for {len(records)} photos; on the clean pairs the marked point is on true damage for "
      f"{HOT_ALL['point_on_gt_damage']} of them (T1 v3 on validation, R9's maps: {T1V3_POINT_ON_DAMAGE})")

# Overlays for the 10 xBD and 10 other clean photos with the most true destroyed pixels (panel 1: the true damage map)
COLOURS = np.array([[45, 45, 45], [60, 180, 75], [255, 200, 0], [230, 25, 75]], np.uint8)
by_name = {r["image"]: r for r in records}


def overlay(rec):
    name, stem = rec["image"], Path(rec["image"]).stem
    pred = read_png(C4Z, name, cv2.IMREAD_GRAYSCALE)
    gt = read_png(MASKS, mask_file[name], cv2.IMREAD_GRAYSCALE)
    danger, safety, priority, _ = survivability_maps(pred)
    background = np.full((*pred.shape, 3), 128, np.uint8)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.4))
    panels = [("True damage map + all markers (red = destroyed)", COLOURS[gt], None, None, None),
              ("Danger (C4's map)", background, danger, "inferno", DISPLAY_MAX),
              ("Safety + 3 safest open areas", background, safety, "viridis", 1.0),
              ("Search priority + top 3 hot spots", background, priority, "inferno", DISPLAY_MAX)]
    for ax, (title, base, m, cmap, vmax) in zip(axes, panels):
        ax.imshow(base)
        if m is not None:
            shown = ax.imshow(m, cmap=cmap, alpha=0.55, vmin=0, vmax=vmax)
            fig.colorbar(shown, ax=ax, fraction=0.046, pad=0.02)
        ax.set_title(title)
        ax.axis("off")
    for ax in (axes[0], axes[2]):
        for a in rec["safe_areas"]:
            ax.plot(*a["xy"], "^", ms=12, color="lime", mec="black")
    for ax in (axes[0], axes[3]):
        for s in rec["hot_spots"]:
            x, y = s["point_xy"]
            ax.plot(x, y, "o", ms=16, mfc="none", mec="cyan", mew=2)
            ax.text(x + 14, y, str(s["rank"]), color="cyan", fontsize=13, weight="bold")
    fig.suptitle(f"{stem}  ({rec['event']}, {rec['source']}: {rec['method']})")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "overlays" / f"{stem}.jpg", dpi=80)
    plt.close(fig)


CHECKED = pd.concat([g.nlargest(N_CHECK, "gt_destroyed_px") for _, g in per_image.groupby("source")])
for n in tqdm(CHECKED["image"], desc="Overlays"):
    overlay(by_name[n])
print(f"✓ {len(CHECKED)} overlays in {OUT_DIR / 'overlays'}")

Hot spots:   0%|          | 0/2065 [00:00<?, ?it/s]

   images_from source  images  hot_spots  on_gt_destroyed  ceiling_from_gt_mask  on_gt_damaged_or_destroyed  point_on_gt_destroyed  point_on_gt_damage  points_within_5px_of_edge  safe_on_gt_building
   clean pairs    all    2007       2775            0.316                 0.486                       0.603                  0.264               0.569                      0.012                0.071
   clean pairs  other     739        770            0.347                 0.507                       0.544                  0.288               0.497                      0.006                0.159
   clean pairs    xBD    1268       2005            0.304                 0.474                       0.626                  0.255               0.596                      0.014                0.039
with destroyed    all     687       1448            0.606                 0.721                       0.865                  0.506               0.827                      0.006                0.054
with 

Overlays:   0%|          | 0/20 [00:00<?, ?it/s]

✓ 20 overlays in D:\CSE499AB_project\Essential_task_delegation\Champion_Task\E7-results\step3\overlays


In [7]:
# ── Cell 7: The table's summary, counting beside, check 6 (Ridita's bridge reads the files), README and config ──
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


# The table: rows by source and kind; on the sharper photos, outlines against E7 step 2's joined blobs
CLASSES = list(CLASS_NAME.values())
TABLE_SUMMARY = (TABLE.groupby(["source", "kind"]).agg(photos=("image", "count"), **{c: (c, "sum") for c in CLASSES},
                                                       **{f"e7_step2_{c}": (f"e7_step2_{c}", "sum") for c in CLASSES})
                 .reset_index())
TABLE_SUMMARY["rows"] = TABLE_SUMMARY[CLASSES].sum(axis=1)
TABLE_SUMMARY["e7_step2_rows"] = TABLE_SUMMARY[[f"e7_step2_{c}" for c in CLASSES]].sum(axis=1)
TABLE_SUMMARY.to_csv(OUT_DIR / "table_summary.csv", index=False)
other = TABLE_SUMMARY[TABLE_SUMMARY["kind"] == "outline"].iloc[0]
MORE_ROWS = other["rows"] / max(other["e7_step2_rows"], 1) - 1
print(TABLE_SUMMARY.to_string(index=False))
print(f"\nOn the sharper photos: {int(other['rows']):,} outline rows against E7 step 2's {int(other['e7_step2_rows']):,} "
      f"joined blobs ({MORE_ROWS:+.1%}), because touching buildings no longer merge")

# Counting beside, deciding nothing: E7 step 2's saved counts, R11's on xBD photos and seed 42's on the others
q = pd.read_csv(S2 / "per_question.csv")
q = q[q["image"].isin(set(names)) & q["R11 | count"].notna() & q["chain_s42 | count"].notna()].copy()
xbd = q["image"].map(source_of) == "xBD"
q["C4 | count"] = np.where(xbd, q["R11 | count"], q["chain_s42 | count"])
q["C4 | correct"] = np.where(xbd, q["R11 | correct"], q["chain_s42 | correct"]).astype(bool)


def count_stats(df, m):
    err = (df[f"{m} | count"] - df["truth"]).abs()
    return {"method": m, "questions": len(df), "all_%": round(100 * df[f"{m} | correct"].mean(), 2),
            "e5_clean_%": round(100 * df.loc[df["e5_clean"] == True, f"{m} | correct"].mean(), 2),
            "mean_abs_error": round(float(err.mean()), 2), "median_abs_error": float(err.median()),
            "within_20%_%": round(100 * float((err <= 0.2 * df["truth"]).mean()), 1)}


def mcnemar(x, y):
    only_x, only_y = int((x & ~y).sum()), int((~x & y).sum())
    return {"only_first_right": only_x, "only_second_right": only_y,
            "p": round(float(binomtest(only_x, only_x + only_y, 0.5).pvalue), 4) if only_x + only_y else 1.0}


COUNTING = pd.DataFrame([count_stats(q, m) for m in ("C4", "R11", "chain_s42")])
COUNTING.to_csv(OUT_DIR / "counting.csv", index=False)
e5 = q[q["e5_clean"] == True]
PAIRED = {f"C4 vs {o}": mcnemar(e5["C4 | correct"], e5[f"{o} | correct"].astype(bool)) for o in ("R11", "chain_s42")}
print("\n" + COUNTING.to_string(index=False))
print(f"Exact McNemar on E5's clean questions: {PAIRED}")
published_counts = pd.read_csv(S2 / "counting_scores.csv").set_index("method")
if not SMOKE_TEST:
    for m in ("R11", "chain_s42"):
        mine = COUNTING.set_index("method").loc[m, "e5_clean_%"]
        assert abs(mine - published_counts.loc[m, "e5_clean_%"]) < 0.01, f"{m}'s counting differs from E7 step 2's"
    print("✓ R11's and seed 42's counting reproduce E7 step 2's on E5's clean questions")

# Check 6: Ridita's bridge, unchanged, reads the new table and hot-spot file
sys.path.insert(0, str(BRIDGE))
import disasteller_inputs as bridge
BRIDGE_TEXT = {}
for src in ("other", "xBD"):
    pool = TABLE[(TABLE["source"] == src) & TABLE["clean"]].sort_values("destroyed", ascending=False)
    name = pool.iloc[0]["image"]
    facts = bridge.format_disaster_facts(name, str(DB), str(OUT_DIR / "hotspots.json"))
    counts = bridge.get_building_counts(name, str(DB))
    row = TABLE.set_index("image").loc[name]
    assert counts["status"] == "ok" and [counts[c] for c in CLASSES] == [int(row[c]) for c in CLASSES], counts
    assert "not found" not in facts and "Building Counts:" in facts and "Survivability Heatmap:" in facts, facts
    BRIDGE_TEXT[src] = facts
    print(f"\n{facts}")
print(f"\n✓ Check 6: Ridita's bridge reads the table and the hot spots for an xBD photo and an other photo")

# README and config
RUN_MIN = round((time.time() - t_start) / 60, 1)
c4_rows = SCORES[(SCORES["scope"] == "clean pairs") & SCORES["method"].str.startswith("C4_")]
expect = {
    "all six checks pass": True,
    "outline rows 5-20% above E7 step 2's joined blobs on the sharper photos": 0.05 <= MORE_ROWS <= 0.20,
    "hot spots' points on true damage 0.70-0.85 on the clean pairs": 0.70 <= HOT_ALL["point_on_gt_damage"] <= 0.85,
    "counting within 1.5 points of R11 and of seed 42 on E5's clean questions":
        all(abs(COUNTING.set_index("method").loc["C4", "e5_clean_%"] - COUNTING.set_index("method").loc[m, "e5_clean_%"]) <= 1.5
            for m in ("R11", "chain_s42")),
}
config = {"task": "E7 step 3", "smoke_test": SMOKE_TEST, "photos": len(names), "clean_pairs_scored": len(clean & set(names)),
          "table_seed": TABLE_SEED, "bias_fixed": BIAS_FIXED, "min_area": MIN_AREA, "rebuild_agreement": round(agreement, 6),
          "t1_v3_settings": {"sigma_px": SIGMA_PX, "damaged_weight": DAMAGED_WEIGHT, "safe_scale_px": SAFE_SCALE_PX,
                             "hot_threshold": HOT_THRESHOLD, "min_spot_px": MIN_SPOT_PX,
                             "point_edge_margin_px": POINT_EDGE_MARGIN_PX, "safe_min_gap_px": SAFE_MIN_GAP_PX,
                             "safe_edge_margin_px": SAFE_EDGE_MARGIN_PX, "point_radius_px": POINT_RADIUS_PX},
          "table_rows": int(n_rows), "rows_by_kind": by_kind, "more_rows_than_e7_step2_on_sharper_photos": round(float(MORE_ROWS), 4),
          "expectation": expect, "versions": {"opencv": cv2.__version__, "numpy": np.__version__, "pandas": pd.__version__,
                                              "python": sys.version.split()[0]},
          "run_time_min": RUN_MIN}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")
SCORES.to_csv(OUT_DIR / "scores.csv", index=False)
CHECK3.to_csv(OUT_DIR / "scores_reproduced.csv", index=False)
checks_md = pd.DataFrame([
    {"check": "1. The file-name rule gives E6's source for every test pair", "result": "passed"},
    {"check": "2. Seed 42's map, rebuilt from the outlines and scores, equals the saved one", "result": f"passed: {agreement:.6%} of pixels"},
    {"check": "3. The scores reproduce the published ones within 0.0005",
     "result": "not applied (smoke test)" if SMOKE_TEST else "passed"},
    {"check": "4. H2's row code rebuilds E7 step 2's table exactly", "result": f"passed on {len(names)} photos"},
    {"check": "5. T1 v3's and the table's self-checks", "result": "passed"},
    {"check": "6. Ridita's bridge reads the table and the hot spots", "result": "passed (an xBD and an other photo)"},
])
(OUT_DIR / "README.md").write_text("\n".join([
    f"# E7 step 3: C4's damage maps, building table and hot spots on the test photos{' (SMOKE TEST)' if SMOKE_TEST else ''}", "",
    "The question, what it makes, the checks and the expectation, fixed before the run, are in the notebook's first cell "
    "([`e7-step3-c4-table-and-hot-spots.ipynb`](../../e7-step3-c4-table-and-hot-spots.ipynb)).", "",
    "**C4** is the damage-map method Abrar chose on 19 Nov: R11, our pixel model, on the coarser xBD photos, and "
    "find-then-grade (SAM 3 outlines each building, our grader labels it) on the sharper photos such as Turkey's.", "",
    f"Run on a PC, no GPU, in {RUN_MIN} min: {len(names)} test photos, {len(clean & set(names))} of them clean pairs "
    "(the ones with trusted true masks, E6).", "",
    "## Checks", "", as_markdown(checks_md), "",
    "## Scores on the clean pairs", "",
    "Reproduced from the saved files:", "", as_markdown(CHECK3), "",
    "C4 per grader seed (the output map and the table use seed 42):", "",
    as_markdown(c4_rows[["method", "miou", "destroyed_iou", "building_iou", "destroyed_precision", "destroyed_recall"]]), "",
    "## The building table (`buildings_c4.sqlite`)", "",
    "H2's columns, read unchanged by Ridita's bridge, plus `kind`: `outline` = one building SAM 3 outlined (the sharper "
    "photos); `patch` = one connected patch of R11's pixels (xBD photos), which can hold several touching buildings. "
    "The `images` table gives each photo's source and method.", "",
    as_markdown(TABLE_SUMMARY[["source", "kind", "photos", *CLASSES, "rows", "e7_step2_rows"]]), "",
    f"On the sharper photos the table has {int(other['rows']):,} buildings, against E7 step 2's {int(other['e7_step2_rows']):,} "
    f"joined blobs for the same grades ({MORE_ROWS:+.1%}): touching buildings no longer merge.", "",
    "## Hot spots and safe areas (`hotspots.json`, T1 v3's method and settings)", "",
    "On the clean pairs, against the true masks:", "", as_markdown(HOT_SUMMARY), "",
    f"- `point_on_gt_damage`: share of hot spots whose marked point is within {POINT_RADIUS_PX} px of a truly damaged or "
    f"destroyed building. T1 v3 on validation, from R9's maps: {T1V3_POINT_ON_DAMAGE}.",
    "- `on_gt_destroyed`: share of hot-spot zones touching truly destroyed pixels; read it against "
    "`ceiling_from_gt_mask`, the same share when the method runs on the true mask.",
    "- `safe_on_gt_building`: share of safe areas that sit on a true building.", "",
    "Photos outside the clean pairs get hot spots too; their ground-truth fields are empty.", "",
    "Examples (overlays of the photos with the most true destroyed pixels):", "",
    *[f"![{n}](overlays/{Path(n).stem}.jpg)\n" for n in list(CHECKED["image"])[:2] + list(CHECKED["image"])[-2:]],
    "## Counting, beside (deciding nothing)", "",
    "From E7 step 2's saved counts (blobs of at least 20 px, the nearest answer option): R11's on xBD photos, seed 42's "
    "on the others.", "", as_markdown(COUNTING), "",
    f"Exact McNemar on E5's clean questions: {PAIRED}.", "",
    "## Against the expectation, fixed before the run", "",
    *[f"- {k}: **{'yes' if v else 'no'}**" for k, v in expect.items()], "",
    "## What Ridita's report writer reads (check 6)", "",
    "```", BRIDGE_TEXT["other"], "```", "",
    "## Files", "",
    "- `buildings_c4.sqlite`: the table (`buildings`, `images`); `table_summary.csv`",
    "- `hotspots.json`, `hotspots_per_image.csv`, `hotspots_summary.csv`",
    "- `scores.csv` (every method and scope), `scores_reproduced.csv` (check 3), `counting.csv`, `config.json`",
    "- `overlays/`: 20 photos to check by eye; `maps/` (256 px danger, safety, priority) and `c4_maps.zip` are git-ignored",
    "",
    "## Next", "",
    "- T2 step 5 (Ridita): point the report bridge at `E7-results/step3/buildings_c4.sqlite` and `E7-results/step3/hotspots.json`. "
    "Where `kind` is `patch`, a row may hold several touching buildings, so a report must not call it one building.",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md, config.json, scores.csv, scores_reproduced.csv, counting.csv and table_summary.csv to {OUT_DIR} "
      f"({RUN_MIN} min)")

source    kind  photos  intact  damaged  destroyed  e7_step2_intact  e7_step2_damaged  e7_step2_destroyed  rows  e7_step2_rows
 other outline     739   53953     1310       2586            43095              1207                2156 57849          46458
   xBD   patch    1326   47652    12839       7099            43904              8761                5473 67590          58138

On the sharper photos: 57,849 outline rows against E7 step 2's 46,458 joined blobs (+24.5%), because touching buildings no longer merge



   method  questions  all_%  e5_clean_%  mean_abs_error  median_abs_error  within_20%_%
       C4       3580  35.34       34.89           12.61               4.0          34.3
      R11       3580  33.02       32.53           13.34               4.0          32.3
chain_s42       3580  32.88       32.70           13.58               4.0          30.7
Exact McNemar on E5's clean questions: {'C4 vs R11': {'only_first_right': 203, 'only_second_right': 120, 'p': 0.0}, 'C4 vs chain_s42': {'only_first_right': 376, 'only_second_right': 299, 'p': 0.0034}}
✓ R11's and seed 42's counting reproduce E7 step 2's on E5's clean questions

=== MODEL-ESTIMATED DAMAGE FACTS FOR IMAGE: hawaii_wildfire_post_638.png ===
(Counts and hot spots come from a damage model's map: estimates, not ground truth. Positions in image pixels.)
- Building Counts:
  * Intact Buildings: 5
  * Damaged Buildings: 0
  * Destroyed Buildings: 217
  * Total Buildings in the table: 222

- Survivability Heatmap:
  * Hot Spots (wher


=== MODEL-ESTIMATED DAMAGE FACTS FOR IMAGE: joplin_tornado_00000147_post_disaster.png ===
(Counts and hot spots come from a damage model's map: estimates, not ground truth. Positions in image pixels.)
- Building Counts:
  * Intact Buildings: 0
  * Damaged Buildings: 22
  * Destroyed Buildings: 216
  * Total Buildings in the table: 238

- Survivability Heatmap:
  * Hot Spots (where the estimated damage is densest):
    - Hot spot #1: marked point at (x=43, y=991), sector: south-west, zone area: 961544 px
  * Safe Open Areas (open ground farthest from the estimated damage):
    - Safe area #1: point at (x=818, y=214), sector: north-east, distance to the nearest damage: 94 px
    - Safe area #2: point at (x=112, y=801), sector: south-west, distance to the nearest damage: 64 px
    - Safe area #3: point at (x=632, y=816), sector: south, distance to the nearest damage: 51 px

✓ Check 6: Ridita's bridge reads the table and the hot spots for an xBD photo and an other photo
✓ Wrote README.md,